# 04 - Analise estatistica e por regime

Carrega o CSV bruto gerado por `src/pipeline/run_all.py` e produz:
1. Tabela-resumo de metricas por modelo.
2. Diagrama de diferenca critica (Demsar 2006) via `autorank`.
3. Analise Bayesiana par a par com ROPE via `baycomp`.
4. Quebra dos resultados por regime (tamanho, numero de classes, proporcao categorica, missing).

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path('..').resolve()))

import pandas as pd

from data.load_tabarena import summarize, RECOMMENDED_TASK_IDS
from src.reports.results_table import summary_by_model, pivot_for_stats
from src.pipeline.stats import demsar_analysis, bayesian_pairwise
from src.pipeline.regime import assign_regimes, aggregate_by_regime

In [2]:
raw = pd.read_csv('../results/raw.csv')
raw.head()

,task_id,dataset,model,auc_ovo,accuracy,g_mean,cross_entropy,fit_time_s,predict_time_s,total_time_s
0,359955,blood-transfusion-service-center,lightgbm,0.769006,0.791111,0.597537,0.466000,0.334508,0.014215,0.348724
1,359955,blood-transfusion-service-center,xgboost,0.727908,0.791111,0.632661,0.511380,1.467943,0.017070,1.485013
2,359955,blood-transfusion-service-center,catboost,0.704624,0.755556,0.557404,0.543412,2.244982,0.006570,2.251552
3,359955,blood-transfusion-service-center,group_model,0.758447,0.791111,0.558472,0.474704,0.406826,0.857361,1.264187
4,359968,churn,lightgbm,0.938793,0.955333,0.849317,0.151686,0.553601,0.016320,0.569921


In [3]:
summary_by_model(raw)

,model,auc_ovo_mean,auc_ovo_std,accuracy_mean,accuracy_std,g_mean_mean,g_mean_std,cross_entropy_mean,cross_entropy_std,total_time_s_mean,total_time_s_std
3,group_model,0.871638,0.096995,0.871907,0.088625,0.663090,0.290883,0.297446,0.170636,529.599407,981.826607
1,autogluon_extreme,0.866640,0.094426,0.865888,0.090090,0.631415,0.292137,0.311109,0.168783,1720.263332,1504.630495
0,autogluon_default,0.866017,0.096221,0.865938,0.091957,0.639523,0.297030,0.312679,0.166682,140.194177,189.241649
4,lightgbm,0.853298,0.103695,0.857472,0.094126,0.617019,0.290395,0.336361,0.171287,0.654392,0.355991
2,catboost,0.852080,0.100434,0.859491,0.090393,0.644086,0.250333,0.384254,0.207655,2.700736,1.105922
5,xgboost,0.849062,0.102937,0.858163,0.091837,0.620421,0.290265,0.422965,0.327900,3.113084,4.068336


In [4]:
pivot = pivot_for_stats(raw, metric='auc_ovo')
demsar = demsar_analysis(pivot, output_dir=Path('../results/figures'))
demsar['ranking']

Tests for normality and homoscedacity are ignored for test selection, forcing nonparametric tests
                   meanrank    median       mad  ci_lower  ci_upper  \
xgboost            5.100000  0.861156  0.090679  0.795847  0.902277   
catboost           4.633333  0.852844  0.096315  0.800159  0.904001   
lightgbm           4.433333  0.848568  0.086868  0.799691  0.906905   
autogluon_default  2.766667  0.865563  0.087421  0.816273   0.91576   
autogluon_extreme  2.600000  0.864174  0.087719  0.817824  0.915455   
group_model        1.466667  0.869467  0.092543  0.821495  0.921781   

                  effect_size   magnitude effect_size_above magnitude_above  
xgboost                   0.0  negligible               0.0      negligible  
catboost            -0.029677  negligible         -0.029677      negligible  
lightgbm            -0.041005  negligible         -0.011937      negligible  
autogluon_default   -0.170169  negligible         -0.127149      negligible  
autogluon_extr

group_model          1.466667
autogluon_extreme    2.600000
autogluon_default    2.766667
lightgbm             4.433333
catboost             4.633333
xgboost              5.100000
Name: mean_rank, dtype: float64

In [5]:
bayes = bayesian_pairwise(pivot, rope=0.01)
bayes

,model_a,model_b,p_a_worse,p_equivalent,p_a_better
0,autogluon_default,autogluon_extreme,0.00000,1.00000,0.00000
1,autogluon_default,catboost,0.81750,0.18250,0.00000
2,autogluon_default,group_model,0.00004,0.99088,0.00908
3,autogluon_default,lightgbm,0.28318,0.71682,0.00000
4,autogluon_default,xgboost,0.98752,0.01248,0.00000
5,autogluon_extreme,catboost,0.76144,0.23856,0.00000
6,autogluon_extreme,group_model,0.00016,0.99234,0.00750
7,autogluon_extreme,lightgbm,0.34688,0.65312,0.00000
8,autogluon_extreme,xgboost,0.96176,0.03824,0.00000
9,catboost,group_model,0.00000,0.00302,0.99698


In [6]:
metadata = assign_regimes(summarize(RECOMMENDED_TASK_IDS))
for col in ['regime_size', 'regime_classes', 'regime_cat_share', 'regime_missing']:
    print(f'\n=== Agregado por {col} ===')
    print(aggregate_by_regime(raw, metadata, regime_col=col, metric_col='auc_ovo'))


=== Agregado por regime_size ===
   regime_size              model      mean       std  count
0        large  autogluon_default  0.843112  0.088551      8
1        large  autogluon_extreme  0.842927  0.089133      8
2        large           catboost  0.836429  0.089739      8
3        large        group_model  0.848803  0.086074      8
4        large           lightgbm  0.834139  0.092811      8
5        large            xgboost  0.833414  0.092481      8
6       medium  autogluon_default  0.876062  0.098824     19
7       medium  autogluon_extreme  0.876157  0.096550     19
8       medium           catboost  0.860152  0.102338     19
9       medium        group_model  0.881600  0.101421     19
10      medium           lightgbm  0.858923  0.110489     19
11      medium            xgboost  0.856098  0.106955     19
12       small  autogluon_default  0.863471  0.125865      3
13       small  autogluon_extreme  0.869594  0.120245      3
14       small           catboost  0.842687  0.1486